# Text atlas

A CPU-friendly 2D map of the 1,250-article historical Wikinews corpus.

Adapted from INRIA's scikit-learn MOOC notebook `dimred_text.ipynb`
(CC BY 4.0, INRIA scikit-learn MOOC contributors):
<https://github.com/INRIA/scikit-learn-mooc/blob/3d1e8cdf7df6675d8a47d352d66b29dfea36587c/notebooks/dimred_text.ipynb>

Corpus: historical Wikinews articles, CC BY 2.5 (Wikinews contributors,
curated by The Mega Rhyme Rhyming Dictionary; see `DATA_LICENSE` and
`DATA_SOURCES.md`).

Pipeline: TF-IDF with English stop-word removal, centered PCA (at most 50
retained components, randomized solver, fixed seed), KMeans in the retained
PCA space. Categories are a display overlay only and never a model input.
This notebook reuses `text_core.py`, the same implementation as `app.py`,
and writes a fresh `results.json` to the current directory.

In [ ]:
import sys
from pathlib import Path

# PROJECT_ROOT is supplied by the runner; fall back to the current directory.
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", Path.cwd())).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import text_core

corpus_path = PROJECT_ROOT / "data" / "wiki_news.csv"
corpus = text_core.load_corpus(corpus_path)
print(f"Loaded {len(corpus)} articles from {corpus_path}")
corpus["category"].value_counts()
corpus.head(3)

In [ ]:
# TF-IDF (English stop words) -> centered PCA (<=50 components, randomized,
# fixed seed) -> KMeans in the retained PCA space.
analysis = text_core.analyze(min_df=5, max_df=0.8, n_clusters=5, seed=42)
analysis["parameters"]

In [ ]:
import json

import pandas as pd

params = analysis["parameters"]
metrics = pd.DataFrame(
    [
        {"metric": "articles", "value": params["n_samples"]},
        {"metric": "vocabulary size", "value": len(analysis["vocabulary"])},
        {"metric": "silhouette (retained PCA space)", "value": round(analysis["silhouette"], 4)},
        {
            "metric": "original variance visible in 2D",
            "value": f"{analysis['displayed_variance'] * 100:.2f}%",
        },
        {
            "metric": "original variance retained by all components",
            "value": f"{analysis['retained_variance'] * 100:.2f}%",
        },
        {"metric": "PCA components retained", "value": params["n_components"]},
        {"metric": "clusters", "value": params["n_clusters"]},
    ]
)
print("Nearby points share similar TF-IDF vocabulary; KMeans clusters are not validated topics. The 2D view drops detail, and silhouette is an internal clustering diagnostic, not an accuracy score.")
metrics

results = {
    "results": {
        "n_articles": params["n_samples"],
        "vocabulary_size": len(analysis["vocabulary"]),
        "silhouette": analysis["silhouette"],
        "displayed_variance": analysis["displayed_variance"],
        "retained_variance": analysis["retained_variance"],
        "n_pca_components": params["n_components"],
        "cluster_sizes": [
            int((analysis["labels"] == cluster).sum())
            for cluster in range(params["n_clusters"])
        ],
        "top_terms": analysis["top_terms"],
    }
}
with open("results.json", "w", encoding="utf-8") as handle:
    json.dump(results, handle, indent=2, allow_nan=False)
print("Wrote results.json to", Path.cwd())